# MatchFormer Epipolar Fine-tuning — Google Colab

Fine-tunes MatchFormer Lite-LA with epipolar supervision on ScanNet.
Checkpoints are saved to **Google Drive** so training survives Colab disconnects.

**Run order (fresh session):** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
**Resume after disconnect:** Cell 1 → 3 → 4 → 7 (skip 2, 5, 6)

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ZIPS    = '/content/drive/MyDrive/final_proj/scannet_zips'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/phase1_gaussian'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Zips dir:       {DRIVE_ZIPS}')
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

## Cell 2 — Clone Repo & Install Dependencies
*(Skip on resume — repo and packages persist for the session)*

In [ ]:
REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone -b develop {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git fetch origin && git checkout -f develop && git pull origin develop

%cd /content/final_proj/MatchFormer



Cloning repo...
Cloning into '/content/final_proj'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 352 (delta 107), reused 88 (delta 43), pack-reused 192 (from 1)
Receiving objects: 100% (352/352), 133.22 MiB | 42.90 MiB/s, done.
Resolving deltas: 100% (190/190), done.
/content/final_proj/MatchFormer


In [3]:
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 17.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.5 MB/s eta 0:00:00
Done.


## Cell 3 — Verify Zips on Drive

In [4]:
import glob, os

%cd /content/final_proj/MatchFormer

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) on Drive:')
for z in zips:
    size_gb = os.path.getsize(z) / 1e9
    print(f'  {os.path.basename(z)}  ({size_gb:.2f} GB)')

/content/final_proj/MatchFormer
Found 11 zip(s) on Drive:
  scene0000_00.zip  (2.24 GB)
  scene0001_00.zip  (0.58 GB)
  scene0002_00.zip  (2.22 GB)
  scene0003_00.zip  (0.60 GB)
  scene0004_00.zip  (0.28 GB)
  scene0005_00.zip  (0.36 GB)
  scene0006_00.zip  (0.77 GB)
  scene0007_00.zip  (0.40 GB)
  scene0008_00.zip  (0.39 GB)
  scene0009_00.zip  (0.48 GB)
  scene0010_00.zip  (0.81 GB)


## Cell 4 — Extract Zips to Local SSD

Copies zips from Drive to Colab's local SSD and extracts them.
Drive has high per-file latency — local SSD is much faster for the dataloader.
Takes ~5–10 min once per session; already-extracted scenes are skipped.

In [5]:
import os, glob, shutil, subprocess, time

LOCAL_DATA = '/content/scannet_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) to extract\n')

failed = []
for i, zip_drive in enumerate(zips, 1):
    scene = os.path.basename(zip_drive).replace('.zip', '')
    dst = os.path.join(LOCAL_DATA, scene)
    prefix = f'[{i}/{len(zips)}] {scene}'

    if os.path.isdir(dst) and len(glob.glob(os.path.join(dst, 'exported/color/*.jpg'))) > 0:
        n = len(glob.glob(os.path.join(dst, 'exported/color/*.jpg')))
        print(f'{prefix} — already extracted ({n} frames), skipping')
        continue

    size_gb = os.path.getsize(zip_drive) / 1e9
    zip_local = f'/content/{scene}.zip'

    t0 = time.time()
    print(f'{prefix} — copying {size_gb:.2f} GB from Drive...', end=' ', flush=True)
    shutil.copy2(zip_drive, zip_local)
    print(f'done ({time.time()-t0:.0f}s). Extracting...', end=' ', flush=True)

    t1 = time.time()
    result = subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA])
    os.remove(zip_local)

    if result.returncode != 0:
        print(f'FAILED (unzip error {result.returncode}) — zip may be corrupted on Drive.')
        failed.append(scene)
    else:
        print(f'done ({time.time()-t1:.0f}s).')

total = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
print(f'\nAll done. {total} frames in {LOCAL_DATA}')
if failed:
    print(f'\nFailed scenes (re-upload zips to Drive): {failed}')

Found 11 zip(s) to extract

[1/11] scene0000_00 — copying 2.24 GB from Drive... done (25s). Extracting... done (13s).
[2/11] scene0001_00 — copying 0.58 GB from Drive... done (11s). Extracting... done (3s).
[3/11] scene0002_00 — copying 2.22 GB from Drive... done (28s). Extracting... done (13s).
[4/11] scene0003_00 — copying 0.60 GB from Drive... done (7s). Extracting... done (4s).
[5/11] scene0004_00 — copying 0.28 GB from Drive... done (3s). Extracting... done (2s).
[6/11] scene0005_00 — copying 0.36 GB from Drive... done (9s). Extracting... done (2s).
[7/11] scene0006_00 — copying 0.77 GB from Drive... done (12s). Extracting... done (5s).
[8/11] scene0007_00 — copying 0.40 GB from Drive... done (9s). Extracting... done (2s).
[9/11] scene0008_00 — copying 0.39 GB from Drive... done (7s). Extracting... done (2s).
[10/11] scene0009_00 — copying 0.48 GB from Drive... done (10s). Extracting... done (3s).
[11/11] scene0010_00 — copying 0.81 GB from Drive... done (10s). Extracting... done 

## Cell 5 — Download Pretrained Weights
*(Skip on resume — weights persist for the session)*

In [6]:
WEIGHTS_PATH  = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
elif os.path.exists(DRIVE_WEIGHTS):
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
else:
    print('WARNING: weights not found. Upload indoor-lite-LA.ckpt to Drive at:')
    print(' ', DRIVE_WEIGHTS)

Copied weights from Drive


## Cell 6 — Verify GPU & Sanity Check

Runs 50 steps on 5 pairs to confirm the pipeline works end-to-end.  
Loss should decrease. Takes ~1 min on L4.

In [8]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

# # Sanity check: 5 pairs, 50 steps, reads from Drive (fine for 5 pairs)
# !python train_finetune.py \
#     --overfit \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
#     --batch 4 \
#     --steps 50

# print('Sanity check done — loss should be decreasing!')

GPU: NVIDIA A100-SXM4-40GB
CUDA: True


## Cell 7 — Phase 1: Scaffolded Training (20k steps)

Epipolar curriculum Phase 1: full-strength Gaussian epipolar mask on confidence matrix.
- **Scenes 0–5**, per-scene 90/10 split (first 90% train, last 10% test)
- **Gaussian decay** (τ=24 ≈ σ₀=3 coarse pixels) applied after dual-softmax
- **LR 3e-5** with CosineAnnealing (eta_min=5e-6)
- Frozen backbone (Block1/2), BN frozen, Block3/4 + FPN head trainable
- `--frame_gap 40` to avoid loss collapse on easy pairs

In [ ]:
import wandb
wandb.login()

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python train_finetune.py \
    --data_dir {LOCAL_DATA} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 20000 \
    --tau 24.0 \
    --batch 4 \
    --workers 4 \
    --lr 3e-5 \
    --eta_min 5e-6 \
    --frame_gap 40 \
    --scenes scene0000_00 scene0001_00 scene0002_00 scene0003_00 scene0004_00 scene0005_00 \
    --split_ratio 0.9 \
    --save_every 2000 \
    --precision bf16 \
    --wandb \
    --wandb_project matchformer-finetune \
    --wandb_run phase1-gaussian-s0to5-20k

## Cell 8 — Pose Estimation Benchmark (Table 2 style)

Runs pose AUC benchmark (AUC @5°, @10°, @20° + matching precision) on:
- **scene0000_00** — seen during training (in-distribution)
- **scene0011_00** — not seen during training (out-of-distribution)

Compares three model variants: Vanilla, Vanilla+Epipolar, Fine-Tuned.
Results saved to `DRIVE_CKPT_DIR`.

In [ ]:
import re, glob, shutil, os, subprocess, time

LOCAL_DATA = '/content/scannet_data'

# ── Pick latest checkpoint ──────────────────────────────────────────────────
step_ckpts = [p for p in glob.glob(f'{DRIVE_CKPT_DIR}/epipolar-*.ckpt')
              if re.search(r'epipolar-step=(\d+)\.ckpt', p)]
if step_ckpts:
    TRAINED_CKPT = max(step_ckpts, key=lambda p: int(re.search(r'epipolar-step=(\d+)', p).group(1)))
else:
    TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'

LOCAL_CKPT = 'model/weights/epipolar-finetuned.ckpt'
os.makedirs('model/weights', exist_ok=True)
shutil.copy(TRAINED_CKPT, LOCAL_CKPT)
print(f'Using checkpoint: {TRAINED_CKPT}')
print(f'Copied to {LOCAL_CKPT}')

# ── Extract scene0011 if not already done ───────────────────────────────────
scene0011_zip = os.path.join(DRIVE_ZIPS, 'scene0011_00.zip')
scene0011_dst = os.path.join(LOCAL_DATA, 'scene0011_00')
if os.path.isdir(scene0011_dst) and len(glob.glob(os.path.join(scene0011_dst, 'exported/color/*.jpg'))) > 0:
    print(f'scene0011_00 already extracted')
elif os.path.exists(scene0011_zip):
    zip_local = '/content/scene0011_00.zip'
    t0 = time.time()
    print('Copying scene0011_00.zip from Drive...', end=' ', flush=True)
    shutil.copy2(scene0011_zip, zip_local)
    print(f'done ({time.time()-t0:.0f}s). Extracting...', end=' ', flush=True)
    subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA])
    os.remove(zip_local)
    print('done.')
else:
    print('WARNING: scene0011_00.zip not found on Drive — skipping scene0011 benchmark')

# ── Run Pose AUC Benchmark ──────────────────────────────────────────────────
# In-distribution (scene0000) + OOD (scene0011), 100 pairs each
!python pose_benchmark.py \
    --num_pairs 100 \
    --data_dir {LOCAL_DATA} \
    --finetuned_ckpt {LOCAL_CKPT} \
    --scenes scene0000_00 scene0011_00 \
    2>&1 | tee pose_benchmark_results.txt

shutil.copy('pose_benchmark_results.txt', f'{DRIVE_CKPT_DIR}/pose_benchmark_results.txt')
print(f'\nResults saved to Drive: {DRIVE_CKPT_DIR}/pose_benchmark_results.txt')